# 02 Plan

What I decided to do after the lectures (`01-lecture-notes.ipynb`), and why. The further research behind each
choice is in `../research/`:

| Research note | What it covers |
|---|---|
| `../research/01-hazard-models.md` | The hazard model families, pros and cons, and fit to our data |
| `../research/02-model-choices.md` | What can vary inside a hazard model: event, risk set, time origin, covariates, baseline |
| `../research/03-clustering.md` | Whether clustering helps, and how to use it to test the metric |
| `../research/references.md` | All sources |

Reading order for my part: **lecture notes → this plan → research notes → modelling (03 to 05)**.

## 1. The question

The group predicts whether a credit-card customer defaults in October 2005 (Taiwan data, Yeh, 2009), each member
with one model, all scored on the same test customers. Oscar does logistic regression, Louis a Markov chain and
decision trees. **My part: hazard models**, plus clustering as a tool for testing the metric.

A hazard model asks, each month, "given this customer has made it this far, what is the chance of the event now?"
(Allison, 1982). It fits our data because the data is monthly: six months of repayment status, bills and payments
(April to September), then the October label.

## 2. Decisions

| Decision | Choice | From |
|---|---|---|
| Core model | **Discrete-time logit hazard**: a logistic regression on a person-month table | Lecture 01.2 (logistic regression is the baseline); research note 01 (the standard credit form: Shumway, 2001; Bellotti & Crook, 2013) |
| Variants | Cloglog link (grouped-time Cox), L1 / L2 penalty, Cox with time-varying covariates | Research note 01: each changes one thing, so the comparison isolates it |
| Not built | Frailty, mixture cure, AFT, recurrent-event Cox | Research note 01: no Python package with covariates for the cure model; AFT shape untestable with 5 time points |
| Event | 2+ months behind for June to September; the real label for October | Research note 02: the data has no default date |
| Risk set | Everyone, every month; last month's status as a covariate | Research note 02: keeps the riskiest customers, so every test customer gets a prediction |
| Covariates | Lagged one month: status (categories), utilisation, payment ratio; log limit, age, sex, education, marriage | Research note 02; status codes -2/-1/0 are categories (`data/README.md`) |
| Choosing between models | AIC, BIC and **10-fold cross-validation with whole customers left out** | Lectures 02.1.P, 02.2 |
| Tuning the penalty | Grouped cross-validation on the training customers | Lectures 02.2, 3.1.P |
| Metrics | AUC with ROC and PR curves; Brier and log loss; calibration | Lecture 01.2, Portfolio 1.2.P |
| Uncertainty | Bootstrap over customers; test set used once | Lecture 05.2 |
| Clustering | k-means on April to September behaviour, with a sensitivity analysis over K and a GMM chosen by BIC | Lectures 04.2, 4.2.P; research note 03 |

## 3. Testing generalisation: the four targets from lecture 02.2

The brief asks when the metric says something about real-world performance. Lecture 02.2 lists what we might want
to generalise to; each needs its own split, so each gets its own test:

| Lecture 02.2 target | Our version | Where |
|---|---|---|
| New but similar source | New customers like the training ones: grouped cross-validation and the shared validation set | `04`, sections 4 and 5 |
| Next point in the same stream | Next month: fit on months up to August, predict September | `04`, section 8 |
| Unexpected, new kind of source | A customer type the model has never seen: leave one cluster out | `05`, section 5 |
| New segment of the same stream | Not tested: we have one six-month window | Limitation |

And from lecture 05.2: the overall AUC can hide poor performance inside groups, so the metric is also reported
**within each cluster** (`05`, section 4).

## 4. Ground rules

* Same shared split as Oscar and Louis (`report/01-Data.ipynb`, seed 2026).
* Everything is chosen on the **training** customers by cross-validation. The validation set is an honest check.
  The test set is used **once**, for the final model, to write `data/output/predictions-alex.csv`.
* Python only. Code is lifted only from documented, citable Python sources (library documentation and examples);
  other sources are cited for the approach and the code is my own. Each notebook lists its sources at the top.

Next: `03-data.ipynb` builds the person-month table.